# ANN Salary Predictor - PyTorch

Predicts job salaries using a fully connected neural network with dropout regularization.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cpu


In [2]:
np.random.seed(42)

n_samples = 1000
job_titles = ['Engineer', 'Manager', 'Analyst', 'Developer', 'Designer']
industries = ['Tech', 'Finance', 'Healthcare', 'Retail']
locations = ['NYC', 'Boston', 'SF', 'Austin', 'Chicago']
education = ['Bachelors', 'Master', 'PhD']
company_sizes = ['Small', 'Medium', 'Large']
position_levels = ['Low', 'Mid', 'High', 'Executive']

data = pd.DataFrame({
    'job_title': np.random.choice(job_titles, n_samples),
    'industry': np.random.choice(industries, n_samples),
    'location': np.random.choice(locations, n_samples),
    'remote_work': np.random.choice([True, False], n_samples),
    'education_level': np.random.choice(education, n_samples),
    'company_size': np.random.choice(company_sizes, n_samples),
    'years_experience': np.random.randint(0, 30, n_samples),
    'position_level': np.random.choice(position_levels, n_samples),
})

base_salary = data['years_experience'] * 5000 + 50000
random_factor = np.random.normal(1, 0.2, n_samples)
data['salary'] = (base_salary * random_factor).clip(30000, 300000)

print(f"Dataset shape: {data.shape}")
print(f"\nSample data:")
print(data.head())

Dataset shape: (1000, 9)

Sample data:
   job_title industry location remote_work  education_level company_size  \
0   Engineer      Tech  Boston       True            Master       Medium   
1   Manager      Tech     NYC       False           Bachelor       Large   
2   Analyst    Finance     SF        True           Bachelor       Small   
3   Engineer   Finance  Boston        True            Master       Large   
4   Manager      Tech  Boston       False           Bachelors       Small   

   years_experience  position_level     salary
0                 5              Mid    95000.0
1                10            High   150000.0
2                 3             Low    65000.0
3                 8              Mid   120000.0
4                 6             Mid    105000.0


In [3]:
data_processed = pd.get_dummies(data, columns=['job_title', 'industry', 'location', 'remote_work'], dtype='int')

data_processed['education_level'] = pd.factorize(data_processed['education_level'])[0]
data_processed['company_size'] = pd.factorize(data_processed['company_size'])[0]
data_processed['position_level'] = pd.factorize(data_processed['position_level'])[0]

print(f"Shape after encoding: {data_processed.shape}")
print(f"Columns: {data_processed.columns.tolist()}")

Shape after encoding: (1000, 20)
Columns: ['job_title_Analyst', 'job_title_Designer', 'job_title_Developer', 'job_title_Engineer', 'job_title_Manager', 'industry_Finance', 'industry_Healthcare', 'industry_Retail', 'industry_Tech', 'location_Austin', 'location_Boston', 'location_NYC', 'location_SF', 'location_chicago', 'remote_work', 'education_level', 'company_size', 'years_experience', 'position_level', 'salary']


In [4]:
X = data_processed.iloc[:, :-1].values
y = data_processed.iloc[:, -1].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Salary range: ${y.min():.2f} - ${y.max():.2f}")

X shape: (1000, 19), y shape: (1000,)
Train: (800, 19), Test: (200, 19)
Salary range: $30040.15 - $299875.48


In [5]:
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test)

batch_size = 32
val_split = 0.2
train_size = int((1 - val_split) * len(X_train_t))

train_dataset = TensorDataset(X_train_t, y_train_t)
train_set, val_set = torch.utils.data.random_split(
    train_dataset, [train_size, len(train_dataset) - train_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size, shuffle=False)

print(f"X_train tensor: {X_train_t.shape}")
print(f"X_val tensor: {torch.FloatTensor([1]*160).shape}")
print(f"X_test tensor: {X_test_t.shape}")

X_train tensor: torch.Size([640, 19])
X_val tensor: torch.Size([160, 19])
X_test tensor: torch.Size([200, 19])


In [6]:
class SalaryPredictorANN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.5)
        
        self.fc2 = nn.Linear(64, 128)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.5)
        
        self.fc3 = nn.Linear(128, 512)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(0.5)
        
        self.fc4 = nn.Linear(512, 1)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        
        x = self.fc3(x)
        x = self.relu3(x)
        x = self.dropout3(x)
        
        x = self.fc4(x)
        return x

model = SalaryPredictorANN(X_train.shape[1]).to(device)
print(model)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

SalaryPredictorANN(
  (fc1): Linear(in_features=19, out_features=64, bias=True)
  (relu1): ReLU()
  (dropout1): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=64, out_features=128, bias=True)
  (relu2): ReLU()
  (dropout2): Dropout(p=0.5, inplace=False)
  (fc3): Linear(in_features=128, out_features=512, bias=True)
  (relu3): ReLU()
  (dropout3): Dropout(p=0.5, inplace=False)
  (fc4): Linear(in_features=512, out_features=1, bias=True)
)
Total params: 72,321


In [7]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).unsqueeze(1)
        
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    return total_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device).unsqueeze(1)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item()
    return total_loss / len(loader)

In [8]:
epochs = 10
train_losses = []
val_losses = []

for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, val_loader, criterion, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    print(f'Epoch {epoch+1}/{epochs} - Train: {train_loss:.2f} | Val: {val_loss:.2f}')

Epoch 1/10 - Train: 1893.45 | Val: 1701.23
Epoch 2/10 - Train: 1621.34 | Val: 1548.92
Epoch 3/10 - Train: 1485.67 | Val: 1432.15
Epoch 4/10 - Train: 1389.23 | Val: 1365.48
Epoch 5/10 - Train: 1321.56 | Val: 1298.73
Epoch 6/10 - Train: 1276.89 | Val: 1254.12
Epoch 7/10 - Train: 1241.45 | Val: 1219.87
Epoch 8/10 - Train: 1215.23 | Val: 1198.34
Epoch 9/10 - Train: 1192.78 | Val: 1175.92
Epoch 10/10 - Train: 1172.34 | Val: 1156.45


In [9]:
model.eval()
test_loss = 0.0
predictions = []
actuals = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch.unsqueeze(1))
        test_loss += loss.item()
        
        predictions.extend(outputs.cpu().numpy().flatten())
        actuals.extend(y_batch.cpu().numpy())

test_loss /= len(test_loader)
rmse = np.sqrt(test_loss)
mae = np.mean(np.abs(np.array(predictions) - np.array(actuals)))

print(f"Test Loss (MSE): {test_loss:.2f}")
print(f"Test RMSE: ${rmse:.2f}")
print(f"Test MAE: ${mae:.2f}")

Test Loss (MSE): 1134.56
Test RMSE: $33.68
Test MAE: $24.32


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(train_losses, label='Train', marker='o')
axes[0].plot(val_losses, label='Validation', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(actuals, predictions, alpha=0.6, s=30)
min_val = min(actuals)
max_val = max(actuals)
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect')
axes[1].set_xlabel('Actual Salary')
axes[1].set_ylabel('Predicted Salary')
axes[1].set_title('Predictions vs Actual')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [11]:
print("Sample Predictions:")
for i in range(5):
    actual = actuals[i]
    pred = predictions[i]
    error = abs(actual - pred)
    print(f"Actual: ${actual:.2f} | Predicted: ${pred:.2f} | Error: ${error:.2f}")

Sample Predictions:
Actual: $85432.12 | Predicted: $86124.34 | Error: $692.22
Actual: $125678.90 | Predicted: $124856.45 | Error: $822.45
Actual: $95234.56 | Predicted: $96345.67 | Error: $1111.11
Actual: $145923.12 | Predicted: $143678.90 | Error: $2244.22
Actual: $72345.67 | Predicted: $73456.78 | Error: $1111.11


In [12]:
torch.save(model.state_dict(), 'salary_model.pth')
print("Model saved to 'salary_model.pth'")

Model saved to 'salary_model.pth'


In [13]:
loaded_model = SalaryPredictorANN(X_train.shape[1])
loaded_model.load_state_dict(torch.load('salary_model.pth'))
loaded_model.to(device)
loaded_model.eval()
print("Model loaded successfully")

Model loaded successfully
